
# Comparing MAP and pure-JAX variational inference

Demonstrates convergence behavior of two inference methods: MAP
(point-estimate via optimization) and pure-JAX geometric variational
inference (native VI). Both are initialized from the same MAP fit, then
evolve independently to show how they explore the posterior. The SFH
panel on the right shows the recovered star-formation history from
each method overlaid on the truth.

Note: VI uses demonstration scale (500 iterations); production requires
2000+ for convergence on larger models.

Reference: Conroy 2013, ARA&A, 51, 393 (SED fitting).


In [ ]:
import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

ssp = tengri.load_ssp()
obs = tengri.Observation(
    photometry=tengri.Photometry.from_names(["sdss_u", "sdss_g", "sdss_r", "sdss_i", "sdss_z"])
)

model = tengri.SEDModel.build(
    ssp,
    observation=obs,
    sfh={"type": "tsnorm", "*": tengri.FREE, "skew": tengri.Fixed(0.3), "trunc": tengri.Fixed(10.0)},
    dust={
        "type": "two_component",
        "*": tengri.FIXED,
        "tau_diff": tengri.Uniform(0.0, 1.5),
        "slope": -0.7,
    },
    redshift=tengri.Fixed(0.1),
)

key = jax.random.PRNGKey(42)
truth = dict(model.spec.sample(key))
truth.update(
    sfh_tsnorm_peak_lbt_gyr=3.0,
    sfh_tsnorm_width_gyr=2.0,
    sfh_tsnorm_log_peak_sfr=1.0,
    dust_tau_diff=0.3,
)
mock = model.mock(truth, snr=20.0, key=key)

forward = tengri.ForwardModel.build(sed=model, observation=obs)

post_map = forward.fit(
    mock.flux_obs, mock.noise, method="map", n_steps=300, verbose=False,
)

post_vi = forward.fit(
    mock.flux_obs, mock.noise, method="native_vi_nonlinear", n_iterations=500, n_samples=3,
    verbose=False,
)

fig, (ax_sfh, ax_sed) = plt.subplots(1, 2, figsize=(12, 4.5))

sfh_truth = model.predict_sfh(truth)
sfh_map = model.predict_sfh(post_map.params)
sfh_vi = model.predict_sfh(post_vi.params)

t_gyr = np.array(sfh_truth.t_gyr)
mask = t_gyr < 5.0

ax_sfh.plot(t_gyr[mask], np.array(sfh_truth.sfr_mean)[mask], "k-", lw=2.0, label="Truth")
ax_sfh.plot(t_gyr[mask], np.array(sfh_map.sfr_mean)[mask], "C3--", lw=1.2, label="MAP")
ax_sfh.plot(t_gyr[mask], np.array(sfh_vi.sfr_mean)[mask], "C0--", lw=1.2, label="VI native")
ax_sfh.set_xlabel("Lookback time [Gyr]")
ax_sfh.set_ylabel(r"SFR [M$_\odot$ yr$^{-1}$]")
ax_sfh.legend(frameon=False, fontsize=9)
ax_sfh.grid(True, alpha=0.3)

sed_truth = model.predict_rest_sed(truth)
sed_map = model.predict_rest_sed(post_map.params)
sed_vi = model.predict_rest_sed(post_vi.params)

wave = np.asarray(sed_truth.wavelength)
z = 0.1
wave_obs = wave * (1.0 + z)

vis = (wave_obs > 2.5e3) & (wave_obs < 1.2e4)
ax_sed.loglog(wave_obs[vis], np.asarray(sed_truth.sed)[vis], "k-", lw=1.2, label="Truth")
ax_sed.loglog(wave_obs[vis], np.asarray(sed_map.sed)[vis], "C3--", lw=1.0, label="MAP")
ax_sed.loglog(wave_obs[vis], np.asarray(sed_vi.sed)[vis], "C0--", lw=1.0, label="VI native")
ax_sed.set_xlabel(r"Observed wavelength $\lambda$ [$\mathrm{\AA}$]")
ax_sed.set_ylabel(r"$F_\nu$ [erg s$^{-1}$ cm$^{-2}$ Hz$^{-1}$]")
ax_sed.legend(frameon=False, fontsize=9)
ax_sed.grid(True, alpha=0.3, which="both")

fig.tight_layout()
fig.savefig("plot_method_comparison.png", dpi=150, bbox_inches="tight")